# PHAS0051 Part B — Adaptive Gaussian fitting
Enter the scattering angle when prompted. All signal, background, smoothed, and mask arrays use the same 511 valid channels.

Place the signal and 10-minute background TXT files alongside this notebook.

Run the notebook to save one `partb_<angle>deg_analysis.png` file in the data folder. It contains the selected fit, its residuals, and the fitting-window scan.


In [ ]:
"""PHAS0051 Part B: angle-configurable adaptive Compton peak fitting.
Place the signal and appropriate background TXT files beside this script/notebook.
Enter any measured angle when prompted; the matching signal filename is found automatically.
"""
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks, peak_widths
from scipy.optimize import curve_fit
from scipy.stats import chi2, norm

FOLDER = Path.cwd()
if not (FOLDER / 'partb_background_10min.txt').is_file():
    matches = list((Path.home() / 'Desktop').glob('*/0051/partb_background_10min.txt'))
    if len(matches) != 1:
        raise FileNotFoundError('Open this notebook from the data folder, or keep one Desktop/*/0051 data folder.')
    FOLDER = matches[0].parent
ANGLE_DEG = None  # Enter any angle when the program starts
SIGNAL_FILE = None  # Automatically detected from angle; optional manual override
BACKGROUND_FILE = FOLDER / 'partb_background_10min.txt'
SIGNAL_SECONDS = 1200
BACKGROUND_SECONDS = 600
SLOPE = 2.2513  # keV/channel, from Part A
INTERCEPT = -36.87  # keV, from Part A
SLOPE_ERROR = 0.0050
INTERCEPT_ERROR = 1.50
E0 = 661.657
ELECTRON_REST_ENERGY = 511.0
SMOOTH_SIGMA = 2
SEARCH_HALF_WIDTH = 40  # search for a local peak near the expected channel
FACTOR_START = 1.0
FACTOR_STEP = 0.25
MAX_FACTOR = 4.0


def read_counts(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path.name}; save the correct spectrum beside this script.')
    data = np.loadtxt(path, skiprows=5, usecols=1)
    if len(data) != 512 or not np.all(np.isfinite(data)):
        raise ValueError(f'{path.name}: expected 512 finite channel counts, got {len(data)}')
    return data


def gaussian(x, height, center, width):
    return height * np.exp(-0.5 * ((x - center) / width) ** 2)


def model(x, height, center, width, baseline, gradient):
    return gaussian(x, height, center, width) + baseline + gradient * (x - 200.0)


def runs_p_value(residual):
    signs = np.asarray(residual) > 0
    n_plus = np.count_nonzero(signs)
    n_minus = len(signs) - n_plus
    if min(n_plus, n_minus) == 0 or len(signs) < 3:
        return 0.0
    runs = 1 + np.count_nonzero(signs[1:] != signs[:-1])
    expected = 1 + 2 * n_plus * n_minus / len(signs)
    variance = (2 * n_plus * n_minus * (2 * n_plus * n_minus - len(signs)) /
                (len(signs)**2 * (len(signs) - 1)))
    if variance <= 0:
        return 0.0
    return float(2 * norm.sf(abs(runs - expected) / np.sqrt(variance)))


def fit_window(channels, net, errors, left, right, detected, peak_width):
    x = channels[left:right + 1].astype(float)
    y = net[left:right + 1]
    u = errors[left:right + 1]
    if len(x) <= 7:
        return None
    baseline = float(np.median(np.r_[y[:max(2, len(y)//6)], y[-max(2, len(y)//6):]]))
    guess = [max(float(np.max(y) - baseline), 1.0), float(detected),
             max(peak_width / 2.355, 2.0), baseline, 0.0]
    lower = [0.0, float(left), 1.0, -np.inf, -np.inf]
    upper = [np.inf, float(right), 70.0, np.inf, np.inf]
    try:
        pars, cov = curve_fit(model, x, y, p0=guess, sigma=u,
                              absolute_sigma=True, bounds=(lower, upper), maxfev=30000)
    except (RuntimeError, ValueError):
        return None
    z = (y - model(x, *pars)) / u
    dof = len(x) - len(pars)
    if dof <= 0:
        return None
    reduced = float(np.sum(z*z) / dof)
    limit = float(chi2.isf(0.05, dof) / dof)
    runs_p = runs_p_value(z)
    # A numerical fit alone does not establish that a Compton peak is detected.
    center_error = float(np.sqrt(max(cov[1, 1], 0)))
    height_error = float(np.sqrt(max(cov[0, 0], 0)))
    return dict(left=left, right=right, x=x, y=y, u=u, pars=pars, cov=cov,
                z=z, reduced=reduced, limit=limit, runs_p=runs_p,
                center_error=center_error, height_error=height_error,
                passed=(reduced <= limit and runs_p >= 0.05))


def resolve_signal_file(angle):
    if SIGNAL_FILE is not None:
        return Path(SIGNAL_FILE)
    token = f'{angle:g}'
    candidates = [
        FOLDER / f'{token}degree_20min.txt',
        FOLDER / f'{token}_degree_20min.txt',
        FOLDER / f'{token}degree20min.txt',
        FOLDER / f'{token}_20min.txt',
    ]
    existing = [p for p in candidates if p.is_file()]
    if len(existing) == 1:
        return existing[0]
    if len(existing) > 1:
        raise ValueError(f'Multiple files for {angle:g} degrees: {existing}; set SIGNAL_FILE explicitly.')
    raise FileNotFoundError(f'No signal file for {angle:g} degrees. Tried: {[p.name for p in candidates]}')


def main(angle=None):
    if angle is None:
        angle = ANGLE_DEG
    if angle is None:
        angle = float(input('Enter scattering angle in degrees (e.g. 50, 60, 75): '))
    angle = float(angle)
    if not np.isfinite(angle) or not 0 < angle <= 180:
        raise ValueError('Angle must be between 0 and 180 degrees.')
    signal_file = resolve_signal_file(angle)
    print(f'Angle: {angle:g} degrees; signal: {signal_file.name}')
    signal = read_counts(signal_file)
    background = read_counts(BACKGROUND_FILE)
    if len(signal) != len(background):
        raise ValueError('Signal and background have different numbers of channels.')
    # Exclude the known anomalous last channel ONCE for ALL arrays.
    signal = signal[:-1]
    background = background[:-1]
    channels = np.arange(len(signal))
    ratio = SIGNAL_SECONDS / BACKGROUND_SECONDS
    net = signal - ratio * background
    errors = np.sqrt(np.maximum(signal + ratio**2 * background, 1.0))
    # Last channel may contain an acquisition artifact; exclude it from search and fitting.
    e_theory = E0 / (1 + E0 / ELECTRON_REST_ENERGY * (1 - np.cos(np.deg2rad(angle))))
    expected = (e_theory - INTERCEPT) / SLOPE
    smoothed = gaussian_filter1d(net, SMOOTH_SIGMA)
    lo = max(2, int(expected - SEARCH_HALF_WIDTH))
    hi = min(len(channels)-2, int(expected + SEARCH_HALF_WIDTH))
    candidates, info = find_peaks(smoothed, prominence=0)
    nearby = candidates[(candidates >= lo) & (candidates <= hi)]
    if len(nearby) == 0:
        raise RuntimeError('No local maximum near predicted channel; inspect spectrum.')
    # Choose by prominence, not by proximity to theory or by absolute counts.
    prominence = dict(zip(candidates, info['prominences']))
    detected = int(max(nearby, key=lambda c: prominence[c]))
    width = float(peak_widths(smoothed, [detected], rel_height=0.5)[0][0])
    width = max(width, 3.0)
    print(f'Theoretical energy: {e_theory:.2f} keV; expected channel: {expected:.2f}')
    print(f'Automatic peak: channel {detected}; half-prominence width: {width:.2f} channels')
    print('factor | window | centre +/- error | reduced chi2 / limit | runs p | decision')
    records = []
    chosen = None
    factor = FACTOR_START
    while factor <= MAX_FACTOR + 1e-8:
        left = max(0, int(np.floor(detected - factor*width)))
        right = min(len(channels)-1, int(np.ceil(detected + factor*width)))
        result = fit_window(channels, net, errors, left, right, detected, width)
        if result is None:
            print(f'{factor:4.2f} | {left:3d}-{right:3d} | fit failed')
            if chosen is not None:
                break
        else:
            result['factor'] = factor
            records.append(result)
            p = result['pars']
            print(f'{factor:4.2f} | {left:3d}-{right:3d} | '
                  f'{p[1]:7.2f} +/- {result["center_error"]:5.2f} | '
                  f'{result["reduced"]:5.3f} / {result["limit"]:5.3f} | '
                  f'{result["runs_p"]:6.3f} | {"PASS" if result["passed"] else "FAIL"}')
            if result['passed']:
                chosen = result
            elif chosen is not None:
                break
        factor = round(factor + FACTOR_STEP, 8)
    if chosen is None:
        print('\nWARNING: No window passed BOTH tests; no validated peak centre is reported.')
        if not records:
            return
        # For diagnostic plotting only, not a validated measurement.
        chosen = min(records, key=lambda r: r['reduced'])
        validated = False
    else:
        validated = True
    mu = float(chosen['pars'][1])
    mu_err = chosen['center_error']
    measured_energy = SLOPE * mu + INTERCEPT
    # Calibration a,b are correlated; no covariance supplied, so show only fit component.
    energy_fit_error = abs(SLOPE) * mu_err
    print(f'\nSelected factor: {chosen["factor"]:.2f}; window: {chosen["left"]}-{chosen["right"]}')
    print(f'Peak centre: {mu:.3f} +/- {mu_err:.3f} channels')
    print(f'Energy: {measured_energy:.2f} +/- {energy_fit_error:.2f} keV (FIT ERROR ONLY)')
    print(f'Theory: {e_theory:.2f} keV; difference: {measured_energy-e_theory:+.2f} keV')
    print('Status:', 'WINDOW TESTS PASSED; peak significance still needs checking' if validated else 'UNVALIDATED DIAGNOSTIC FIT')
    if chosen['height_error'] > 0:
        print(f'Indicative amplitude / SE: {chosen["pars"][0]/chosen["height_error"]:.2f} (not a formal discovery significance)')
    print('Calibration error needs slope/intercept COVARIANCE for correct propagation.')
    if chosen['factor'] >= MAX_FACTOR - 1e-8:
        print('CAUTION: Selected factor reached scan maximum; this does not prove it is optimal.')

    # One figure per angle: fitted peak, its residuals, and the window scan.
    fig = plt.figure(figsize=(12, 7), layout='constrained')
    grid = fig.add_gridspec(2, 2, width_ratios=[1.65, 1], height_ratios=[2.2, 1])
    ax_fit = fig.add_subplot(grid[0, 0])
    ax_res = fig.add_subplot(grid[1, 0], sharex=ax_fit)
    ax_scan = fig.add_subplot(grid[:, 1])

    x, y, u, pars = chosen['x'], chosen['y'], chosen['u'], chosen['pars']
    dense = np.linspace(x[0], x[-1], 600)
    ax_fit.errorbar(x, y, yerr=u, fmt='.', color='black', alpha=0.7,
                    markersize=3, linewidth=0.6, label='Net counts +/- counting error')
    ax_fit.plot(dense, model(dense, *pars), color='crimson', linewidth=2,
                label='Gaussian + local continuum')
    ax_fit.plot(dense, pars[3] + pars[4] * (dense - 200.0),
                color='green', linestyle='--', label='Local continuum')
    ax_fit.axvline(mu, color='purple', linestyle=':', label=f'Fitted centre {mu:.2f}')
    ax_fit.set_ylabel('Net counts per channel')
    ax_fit.set_title(f'Selected fit: channels {chosen["left"]}-{chosen["right"]}')
    ax_fit.legend(fontsize=8)

    ax_res.plot(x, chosen['z'], '.', color='black', markersize=3)
    ax_res.axhline(0, color='gray', linewidth=1)
    for level in (-2, 2):
        ax_res.axhline(level, color='gray', linestyle=':', linewidth=1)
    ax_res.set_xlabel('Channel')
    ax_res.set_ylabel('Residual / error')
    ax_res.set_title(f'Residual score R = {chosen["reduced"]:.3f}; limit = {chosen["limit"]:.3f}')

    factors = [record['factor'] for record in records]
    ax_scan.plot(factors, [record['reduced'] for record in records], 'o-',
                 label='Residual score R')
    ax_scan.plot(factors, [record['limit'] for record in records], 's--',
                 markersize=4, color='gray', label='95% upper limit')
    ax_scan.scatter([chosen['factor']], [chosen['reduced']], s=100,
                    color='green', edgecolors='black', zorder=5, label='Selected')
    next_failed = next((record for record in records
                        if record['factor'] > chosen['factor'] and not record['passed']), None)
    if next_failed is not None:
        ax_scan.scatter([next_failed['factor']], [next_failed['reduced']], s=100,
                        color='red', edgecolors='black', zorder=5, label='Next range failed')
    elif chosen['factor'] >= MAX_FACTOR - 1e-8:
        ax_scan.text(0.03, 0.03, 'Scan limit reached; no wider range tested',
                     transform=ax_scan.transAxes, fontsize=8,
                     bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'})
    ax_scan.set_xlabel('Window coefficient k')
    ax_scan.set_ylabel('Reduced weighted residual score')
    ax_scan.set_title('Fitting-window scan')
    ax_scan.legend(fontsize=8)

    for ax in (ax_fit, ax_res, ax_scan):
        ax.grid(alpha=0.2)
    fig.suptitle(f'Part B {angle:g} deg: scattered photopeak analysis')
    output_path = FOLDER / f'partb_{angle:g}deg_analysis.png'
    fig.savefig(output_path, dpi=180)
    print('One analysis figure saved:', output_path)
    plt.show()
    plt.close(fig)

if __name__ == '__main__':
    main()
